In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [0]:
data = [
    (1, 5, 100.0, 70.0, "Laptop", "John"),
    (2, 3, 200.0, 150.0, "Mobile", "Alice"),
    (3, 10, 50.0, 30.0, "Headphones", "Bob"),
    (4, 7, 120.0, 90.0, "Tablet", "John"),
    (5, 15, 20.0, 10.0, "Mouse", "Alice"),
    (6, 8, 80.0, 60.0, "Keyboard", "Bob"),
    (7, 4, 150.0, 100.0, "Monitor", "David"),
    (8, 20, 15.0, 8.0, "USB Cable", "Alice")
]

columns = ["sales_id", "sales_quantity", "selling_price", "cost", "product_name", "customer_name"]

df_sales = spark.createDataFrame(data, columns)

df_sales.display()

sales_id,sales_quantity,selling_price,cost,product_name,customer_name
1,5,100.0,70.0,Laptop,John
2,3,200.0,150.0,Mobile,Alice
3,10,50.0,30.0,Headphones,Bob
4,7,120.0,90.0,Tablet,John
5,15,20.0,10.0,Mouse,Alice
6,8,80.0,60.0,Keyboard,Bob
7,4,150.0,100.0,Monitor,David
8,20,15.0,8.0,USB Cable,Alice


### Calculate Revenue and Profit

In [0]:
df_sales_calc = df_sales .withColumn("revenue", col("sales_quantity") * col("selling_price")) .withColumn("profit", (col("selling_price") - col("cost")) * col("sales_quantity"))
df_sales_calc.display()


sales_id,sales_quantity,selling_price,cost,product_name,customer_name,revenue,profit
1,5,100.0,70.0,Laptop,John,500.0,150.0
2,3,200.0,150.0,Mobile,Alice,600.0,150.0
3,10,50.0,30.0,Headphones,Bob,500.0,200.0
4,7,120.0,90.0,Tablet,John,840.0,210.0
5,15,20.0,10.0,Mouse,Alice,300.0,150.0
6,8,80.0,60.0,Keyboard,Bob,640.0,160.0
7,4,150.0,100.0,Monitor,David,600.0,200.0
8,20,15.0,8.0,USB Cable,Alice,300.0,140.0


### Store as Delta Table

In [0]:
df_sales_calc.write .format("delta") .saveAsTable("sales_delta_table")


### Highest Cost per Product

In [0]:
highest_cost_per_product = df_sales_calc.groupBy("product_name") .agg(max("cost").alias("highest_cost"))
highest_cost_per_product.display()


product_name,highest_cost
Laptop,70.0
Mobile,150.0
Tablet,90.0
Headphones,30.0
Mouse,10.0
Keyboard,60.0
USB Cable,8.0
Monitor,100.0


### Customer Who Bought Highest Number of Items

In [0]:
customer_highest_items = df_sales_calc.groupBy("customer_name").agg(sum("sales_quantity").alias("total_items_purchased")).orderBy(col("total_items_purchased").desc())

customer_highest_items.display()

customer_name,total_items_purchased
Alice,38
Bob,18
John,12
David,4
